In [11]:
import gymnasium as gym
import torch
import torch.nn as nn
import numpy as np
import random
import matplotlib.pyplot as plt
train_env = gym.make('MountainCar-v0', render_mode=None)
test_env = gym.make('MountainCar-v0', render_mode="rgb_array")

episodes = 1500
discount = 0.95
step_size = 0.01
batch_size = 64
epsilon = 1
value_losses = []
policy_losses = []

class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(train_env.observation_space.shape[0], 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 16)
        self.relu = nn.ReLU()
        self.fc3 = nn.Linear(16, train_env.action_space.n)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.softmax(x)
        return x

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(train_env.observation_space.shape[0], 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 8)
        self.relu = nn.ReLU()
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

policy_net = Actor()
value_net = Critic()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy_net.to(device)
value_net.to(device)
criterion = nn.MSELoss()
policy_optimizer = torch.optim.Adam(policy_net.parameters(), lr=step_size)
value_optimizer = torch.optim.Adam(value_net.parameters(), lr=step_size)
buffer = []

def show_ep():
    frame = test_env.render()
    plt.imshow(frame)
    plt.axis("off")
    plt.show()

for ep in range(episodes):
    done = False
    obs, info = train_env.reset()
    t = 0
    avg_policy_loss = 0
    avg_value_loss = 0
    epsilon = max(0.05, 1-ep/episodes)
    if ep>0:
        print(ep, policy_losses[-1], value_losses[-1])

    while not done:
        #show_ep()
        t += 1
        pred = policy_net(torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0))
        if np.random.rand() < epsilon:
            action = train_env.action_space.sample()
        else:
            action = torch.distributions.Categorical(pred).sample().item()

        save_obs = obs
        obs, reward, terminated, truncated, info = train_env.step(action)
        reward += abs(obs[1])
        done = t>500 or terminated

        buffer.append((save_obs, reward, action, obs, done))
        buffer = buffer[-int(1e4):]

        if len(buffer)>=batch_size:
            batch = random.sample(buffer, batch_size)

            states, rewards, actions, next_states, dones = zip(*batch)
            states = torch.tensor(np.array(states), dtype=torch.float32, device=device)
            rewards = torch.tensor(rewards, dtype=torch.float32, device=device)
            actions = torch.tensor(actions, dtype=torch.int64, device=device)
            next_states = torch.tensor(np.array(next_states), dtype=torch.float32, device=device)
            dones = torch.tensor(dones, dtype=torch.float32, device=device)

            with torch.no_grad():
                td_target = rewards + discount * value_net(next_states).squeeze(1) * (1 - dones)
            V_s = value_net(states).squeeze(1)
            td_error = td_target - V_s
            advantages = (td_error - td_error.mean()) / (td_error.std() + 1e-8)

            critic_loss = criterion(V_s, td_target)
            value_optimizer.zero_grad()
            critic_loss.backward()
            value_optimizer.step()
            avg_value_loss = (avg_value_loss*(t-1)+np.mean(critic_loss.item()))/t

            action_probs = torch.log_softmax(policy_net(states), dim=1).gather(1, actions.unsqueeze(1)).squeeze(1)
            actor_loss = -(action_probs * advantages.detach()).mean()
            policy_optimizer.zero_grad()
            actor_loss.backward()
            policy_optimizer.step()
            avg_policy_loss = (avg_policy_loss*(t-1)+np.mean(actor_loss.item()))/t

    value_losses.append(critic_loss.item())
    policy_losses.append(actor_loss.item())

    if ep%500==0:
        done = False
        obs, info = test_env.reset()
        while not done:
            pred = policy_net(torch.tensor(obs).unsqueeze(0))
            obs, reward, terminated, truncated, info = test_env.step(torch.argmax(pred).item())
            done = truncated or terminated
            show_ep()

window = 50
x = range(0,episodes,window)
print('Value Network Loss')
avg_value = [float(np.mean(value_losses[i:i+window])) for i in x]
plt.plot(x,avg_value)
plt.show()
print('Policy Network Loss')
avg_policy = [float(np.mean(policy_losses[i:i+window])) for i in x]
plt.plot(x,avg_policy)
plt.show()

done = False
obs, info = test_env.reset()
while not done:
    pred = policy_net(torch.tensor(obs).unsqueeze(0))
    obs, reward, terminated, truncated, info = test_env.step(torch.argmax(pred).item())
    done = truncated or terminated
    show_ep()

Output hidden; open in https://colab.research.google.com to view.